# Load data

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from funcs.load_data import load_data, get_frequency
from funcs.stat_models import sma_predict, ets_predict, sarima_predict
from funcs.backtest import walk_forward_validation, Backtester
from funcs.metrics import evaluate_metrics
from funcs.plots import plot_forecast_vs_actual, plot_metrics, plot_equity_curves
import plotly.graph_objects as go

# Настройки
TEST_START = "2025-01-01"  # Начало тестового периода
THRESHOLD = 0.005          # Порог торговой стратегии

# Загрузка данных
data = load_data("./data")
print("Доступные активы:", list(data.keys()))
for name, series in data.items():
    print(f"{name}: {len(series)} наблюдений, {series.index.min().date()} - {series.index.max().date()}")

Доступные активы: ['SBER', 'YNDX', 'BTC', 'ETH']
SBER: 2343 наблюдений, 2017-01-03 - 2026-04-30
YNDX: 2313 наблюдений, 2017-01-03 - 2026-04-30
BTC: 3407 наблюдений, 2017-01-01 - 2026-04-30
ETH: 3095 наблюдений, 2017-11-09 - 2026-04-30


# Статистические модели

In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning
warnings.filterwarnings('ignore', category=ValueWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Горизонты
HORIZONS = (1, 7, 21)

# Словари для хранения результатов
all_results = {}       # (asset, model) -> результаты прогнозов
all_metrics = {}       # (asset, model) -> метрики
all_backtests = {}     # (asset, model, horizon) -> DataFrame бэктеста

# Для каждого актива
for asset_name in data:
# for asset_name in ['ETH']:
    series = data[asset_name]
    freq = get_frequency(series)
    
    seasonal_periods = 5 if freq == 'B' else 7  # для акций 5, для крипто 7
    print(f"\n=== {asset_name} (частота {freq}, сезонный период {seasonal_periods}) ===")

    # Модели
    models = {
        'SMA_20': lambda train, steps: sma_predict(train, steps, window=20),
        'ETS': lambda train, steps: ets_predict(train, steps, seasonal_periods=seasonal_periods),
        'SARIMA': lambda train, steps: sarima_predict(train, steps, seasonal_periods=seasonal_periods),
    }
    for model_name, model_func in models.items():
        print(f"\n  Модель: {model_name}")
        for h in HORIZONS:
            df_pred = walk_forward_validation(
                series, model_func, test_start=TEST_START, horizon=h, logs=False
            )
            all_results[(asset_name, model_name, h)] = df_pred
            df_pred.to_excel(f'results/1_{model_name}/predictions/{asset_name}_preds_{model_name}_{h}.xlsx')

            metrics = evaluate_metrics(df_pred)
            all_metrics[(asset_name, model_name, h)] = metrics
            print(f"      RMSE={metrics['RMSE']:.4f}, MAE={metrics['MAE']:.4f}, MAPE={metrics['MAPE']:.2f}%")

            bt = Backtester(
                predictions_df=all_results[(asset_name, model_name, h)],
                initial_capital=100_000,
                threshold=0.005,
                commission=0.001,
            )
            results_df, trades_df, fig = bt.run()

            results_df.to_excel(f'results/1_{model_name}/backtest/{asset_name}_backtest_{model_name}_{h}.xlsx')
            trades_df.to_excel(f'results/1_{model_name}/backtest/{asset_name}_trades_{model_name}_{h}.xlsx')
            fig.write_html(f'results/1_{model_name}/plots/{asset_name}_{model_name}_{h}.html')

            # fig.show()


=== SBER (частота B, сезонный период 5) ===

  Модель: SMA_20


  H=1: 100%|██████████| 335/335 [00:00<00:00, 2948.44window/s]


      RMSE=9.0438, MAE=6.5219, MAPE=2.16%


  H=7: 48window [00:00, 783.09window/s]          


      RMSE=10.2332, MAE=7.3958, MAPE=2.45%


...

      RMSE=112.9465, MAE=78.3653, MAPE=2.73%


  H=7: 70window [09:21,  8.03s/window]                    


      RMSE=225.4066, MAE=157.7521, MAPE=5.49%


  H=21: 24window [03:27,  8.63s/window]                    

      RMSE=363.7487, MAE=269.2284, MAPE=9.78%


In [4]:
import os

metrics_rows = []
for (asset, model, h), m in all_metrics.items():
    metrics_rows.append({
        'Актив': asset,
        'Модель': model,
        'Горизонт': h,
        'RMSE': m['RMSE'],
        'MAE': m['MAE'],
        'MAPE': m['MAPE']
    })
metrics_df = pd.DataFrame(metrics_rows)

# Сохраняем в Excel
os.makedirs('results/metrics', exist_ok=True)
metrics_df.to_excel('results/metrics/all_metrics.xlsx', index=False)
print(metrics_df)

   Актив  Модель  Горизонт         RMSE          MAE       MAPE
0   SBER  SMA_20         1     9.043790     6.521924   2.158489
1   SBER  SMA_20         7    10.233164     7.395802   2.449083
2   SBER  SMA_20        21    12.785896     9.249358   3.075230
3   SBER     ETS         1     3.852055     2.664824   0.879559
4   SBER     ETS         7     6.376254     4.554804   1.498933
5   SBER     ETS        21    10.347734     7.244185   2.360697
6   SBER  SARIMA         1     3.832035     2.634132   0.869517
7   SBER  SARIMA         7     6.344052     4.571138   1.504055
8   SBER  SARIMA        21    10.122976     7.053927   2.300619
9   YNDX  SMA_20         1   170.488719   138.700970   3.264664
10  YNDX  SMA_20         7   192.691832   155.737552   3.664853
11  YNDX  SMA_20        21   254.317371   206.445795   4.863140
12  YNDX     ETS         1    72.438102    55.253479   1.306490
13  YNDX     ETS         7   141.998731   106.988052   2.530576
14  YNDX     ETS        21   193.276043 

# ML models

In [ ]:
import pandas as pd
import numpy as np
import os
from funcs.load_data import load_data
from funcs.backtest import walk_forward_validation, Backtester
from funcs.features_create import create_features
from funcs.metrics import evaluate_metrics
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# Настройки
TEST_START = "2025-01-01"
HORIZONS = (1, 7, 21)

# Загрузка данных
data = load_data("./data")
print("Активы:", list(data.keys()))

# Конфигурации моделей
ML_MODELS = {
    'RandomForest': {
        'model_class': RandomForestRegressor,
        'model_params': {'n_estimators': 200, 'max_depth': 10, 'random_state': 42, 'n_jobs': -1}
    },
    'CatBoost': {
        'model_class': CatBoostRegressor,
        'model_params': {'iterations': 200, 'depth': 6, 'random_seed': 42, 'verbose': False, 'thread_count': -1}
    }
}

all_results = {}
all_metrics = {}

for asset_name in data:
    series = data[asset_name]
    print(f"\n=== {asset_name} ===")
    for model_name, cfg in ML_MODELS.items():
        print(f"  Модель: {model_name}")
        for h in HORIZONS:
            # Генерация признаков
            X, y = create_features(series, horizon=h)

            # Конфиг для ML-режима
            ml_config = {
                'X': X,
                'y': y,
                'model_class': cfg['model_class'],
                'model_params': cfg['model_params']
            }

            # Вызов универсальной Walk-Forward (без передачи model_func)
            df_pred = walk_forward_validation(
                series,
                ml_config=ml_config,
                test_start=TEST_START,
                horizon=h
            )
            all_results[(asset_name, model_name, h)] = df_pred

            # Метрики
            metrics = evaluate_metrics(df_pred)
            all_metrics[(asset_name, model_name, h)] = metrics
            print(f"    H={h}: RMSE={metrics['RMSE']:.4f}, MAE={metrics['MAE']:.4f}, MAPE={metrics['MAPE']:.2f}%")

            # Бэктест
            bt = Backtester(
                predictions_df=all_results[(asset_name, model_name, h)],
                initial_capital=100_000,
                threshold=0.005,
                commission=0.001,
            )
            results_df, trades_df, fig = bt.run()

            # Сохранение результатов
            base_dir = f'results/2_{model_name}'
            os.makedirs(f'{base_dir}/predictions', exist_ok=True)
            os.makedirs(f'{base_dir}/backtest', exist_ok=True)
            os.makedirs(f'{base_dir}/plots', exist_ok=True)

            df_pred.to_excel(f'{base_dir}/predictions/{asset_name}_preds_{model_name}_{h}.xlsx')
            results_df.to_excel(f'{base_dir}/backtest/{asset_name}_backtest_{model_name}_{h}.xlsx')
            trades_df.to_excel(f'{base_dir}/backtest/{asset_name}_trades_{model_name}_{h}.xlsx')
            fig.write_html(f'{base_dir}/plots/{asset_name}_{model_name}_{h}.html')

# Сохранение сводной таблицы метрик
metrics_rows = []
for (asset, model, h), m in all_metrics.items():
    metrics_rows.append({
        'Актив': asset,
        'Модель': model,
        'Горизонт': h,
        'RMSE': m['RMSE'],
        'MAE': m['MAE'],
        'MAPE': m['MAPE']
    })
metrics_df = pd.DataFrame(metrics_rows)
os.makedirs('results/metrics', exist_ok=True)
metrics_df.to_excel('results/metrics/ml_metrics.xlsx', index=False)
print("\nИтоговая таблица метрик ML:")
print(metrics_df)

Активы: ['SBER', 'YNDX', 'BTC', 'ETH']

=== SBER ===
  Модель: RandomForest


  H=1: 100%|█████████▉| 334/335 [03:18<00:00,  1.68window/s]


    H=1: RMSE=3.9213, MAE=2.8010, MAPE=0.92%


  H=7: 100%|██████████| 47/47 [00:24<00:00,  1.88window/s]


    H=7: RMSE=10.2684, MAE=7.6630, MAPE=2.52%


...

    H=1: RMSE=125.4416, MAE=91.7532, MAPE=3.20%


  H=7: 100%|██████████| 69/69 [00:26<00:00,  2.65window/s]


    H=7: RMSE=286.5485, MAE=211.6909, MAPE=7.47%


  H=21: 100%|██████████| 23/23 [00:08<00:00,  2.64window/s]

    H=21: RMSE=520.6455, MAE=408.7013, MAPE=15.30%

Итоговая таблица метрик ML:
   Актив        Модель  Горизонт          RMSE          MAE       MAPE
0   SBER  RandomForest         1      3.921289     2.800994   0.923692
1   SBER  RandomForest         7     10.268439     7.662966   2.522641
2   SBER  RandomForest        21     16.563295    12.660689   4.150383
3   SBER      CatBoost         1      4.548833     3.266814   1.076415
4   SBER      CatBoost         7     11.051576     8.487318   2.795760
5   SBER      CatBoost        21     19.697358    14.596197   4.790755
6   YNDX  RandomForest         1     78.833587    60.531399   1.427182
7   YNDX  RandomForest         7    204.656080   160.937905   3.737180
8   YNDX  RandomForest        21    305.214729   252.073601   5.805518
9   YNDX      CatBoost         1     83.656235    64.616782   1.517864
10  YNDX      CatBoost         7    208.572098   164.483656   3.825847
11  YNDX      CatBoost        21    328.753724   264.317446   6.1063

# Нейросети

In [2]:
from funcs.nn_models import nn_predict

import pandas as pd
import numpy as np
import os
from funcs.load_data import load_data
from funcs.backtest import walk_forward_validation, Backtester
from funcs.features_create import create_features
from funcs.metrics import evaluate_metrics
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# Настройки
TEST_START = "2025-01-01"
HORIZONS = (1, 7, 21)

# Загрузка данных
data = load_data("./data")
print("Активы:", list(data.keys()))

NN_MODELS = {
    'LSTM': lambda train, steps: nn_predict('lstm', train, steps, lookback=20),
    'CNN':  lambda train, steps: nn_predict('cnn', train, steps, lookback=20),
}

all_results = {}
all_metrics = {}

for asset_name in data:
    series = data[asset_name]
    print(f"\n=== {asset_name} ===")
    for model_name, model_func in NN_MODELS.items():
        print(f"\n  Модель: {model_name}")
        for h in HORIZONS:
            df_pred = walk_forward_validation(
                series, model_func, test_start=TEST_START, horizon=h, logs=False
            )
            all_results[(asset_name, model_name, h)] = df_pred

            # Метрики
            metrics = evaluate_metrics(df_pred)
            all_metrics[(asset_name, model_name, h)] = metrics
            print(f"    H={h}: RMSE={metrics['RMSE']:.4f}, MAE={metrics['MAE']:.4f}, MAPE={metrics['MAPE']:.2f}%")

            # Бэктест
            bt = Backtester(
                predictions_df=all_results[(asset_name, model_name, h)],
                initial_capital=100_000,
                threshold=0.005,
                commission=0.001,
            )
            results_df, trades_df, fig = bt.run()

            # Сохранение результатов
            base_dir = f'results/3_{model_name}'
            os.makedirs(f'{base_dir}/predictions', exist_ok=True)
            os.makedirs(f'{base_dir}/backtest', exist_ok=True)
            os.makedirs(f'{base_dir}/plots', exist_ok=True)

            df_pred.to_excel(f'{base_dir}/predictions/{asset_name}_preds_{model_name}_{h}.xlsx')
            results_df.to_excel(f'{base_dir}/backtest/{asset_name}_backtest_{model_name}_{h}.xlsx')
            trades_df.to_excel(f'{base_dir}/backtest/{asset_name}_trades_{model_name}_{h}.xlsx')
            fig.write_html(f'{base_dir}/plots/{asset_name}_{model_name}_{h}.html')


Активы: ['SBER', 'YNDX', 'BTC', 'ETH']

=== SBER ===

  Модель: LSTM


  H=1: 100%|██████████| 335/335 [1:31:43<00:00, 16.43s/window]


    H=1: RMSE=9.2884, MAE=7.0795, MAPE=2.31%


  H=7: 48window [11:54, 14.88s/window]                    


    H=7: RMSE=19.8321, MAE=12.5873, MAPE=4.13%


...

    H=1: RMSE=350.6508, MAE=286.1727, MAPE=10.29%


  H=7: 70window [02:36,  2.24s/window]                    


    H=7: RMSE=416.8011, MAE=333.6308, MAPE=11.88%


  H=21: 24window [00:49,  2.07s/window]                    


    H=21: RMSE=565.4075, MAE=448.7769, MAPE=17.08%


In [3]:
# Сохранение сводной таблицы метрик
metrics_rows = []
for (asset, model, h), m in all_metrics.items():
    metrics_rows.append({
        'Актив': asset,
        'Модель': model,
        'Горизонт': h,
        'RMSE': m['RMSE'],
        'MAE': m['MAE'],
        'MAPE': m['MAPE']
    })
metrics_df = pd.DataFrame(metrics_rows)
os.makedirs('results/metrics', exist_ok=True)
metrics_df.to_excel('results/metrics/nn_metrics.xlsx', index=False)
print("\nИтоговая таблица метрик ML:")
print(metrics_df)


Итоговая таблица метрик ML:
   Актив Модель  Горизонт          RMSE           MAE       MAPE
0   SBER   LSTM         1      9.288421      7.079495   2.311372
1   SBER   LSTM         7     19.832114     12.587266   4.129944
2   SBER   LSTM        21     35.712231     23.842107   7.800886
3   SBER    CNN         1     16.140149     12.200873   3.973730
4   SBER    CNN         7     18.367108     13.701268   4.463882
5   SBER    CNN        21     21.960696     16.858952   5.491077
6   YNDX   LSTM         1    206.659081    161.548711   3.794682
7   YNDX   LSTM         7    289.016561    221.057300   5.160450
8   YNDX   LSTM        21    354.685502    266.326930   6.256349
9   YNDX    CNN         1    272.535194    222.396419   5.184005
10  YNDX    CNN         7    288.520568    228.092844   5.306575
11  YNDX    CNN        21    411.349663    361.030907   8.434346
12   BTC   LSTM         1  16909.175818  14513.292058  14.278653
13   BTC   LSTM         7  20829.416731  18014.635499  17.831

In [2]:
from funcs.load_data import load_data
from funcs.nn_models import nn_predict

data = load_data("./data")
series = data["SBER"]

# Воспроизводим train, который был на первом окне (до 2025-01-03)
train = series[:pd.Timestamp("2025-01-03") - pd.Timedelta(days=1)].copy()
preds = nn_predict("lstm", train, steps=1, lookback=20)
print(preds)

[286.46770613]


In [4]:
from funcs.load_data import load_data
from funcs.nn_models import make_features, train_model, LSTMPredictor
import torch
import pandas as pd

data = load_data("./data")
train = data["SBER"][:pd.Timestamp("2025-01-03") - pd.Timedelta(days=1)].copy()
print(f"Длина train: {len(train)}")

X, y, scaler, df_feat = make_features(train, lookback=20)
print(f"Форма X: {X.shape}, y: {y.shape}")
print(f"Содержит ли X NaN: {np.isnan(X).any()}")
print(f"Содержит ли y NaN: {np.isnan(y).any()}")

if len(X) > 9:
    val_size = max(1, int(0.2 * len(X)))
    X_train, X_val = X[:-val_size], X[-val_size:]
    y_train, y_val = y[:-val_size], y[-val_size:]
    print(f"Train size: {len(X_train)}, Val size: {len(X_val)}")

    # Попробуем обучить одну эпоху без early stopping, чтобы увидеть loss
    model = LSTMPredictor(input_size=X.shape[2])
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.MSELoss()
    X_tensor = torch.from_numpy(X_train).float()
    y_tensor = torch.from_numpy(y_train).float()
    optimizer.zero_grad()
    preds = model(X_tensor)
    loss = criterion(preds, y_tensor)
    print(f"Train loss первый батч: {loss.item()}")
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_preds = model(torch.from_numpy(X_val).float())
        val_loss = criterion(val_preds, torch.from_numpy(y_val).float()).item()
    print(f"Val loss: {val_loss}")
else:
    print("Слишком мало данных – меньше 10 окон.")

Длина train: 2007
Форма X: (1986, 20, 19), y: (1986,)
Содержит ли X NaN: False
Содержит ли y NaN: False
Train size: 1589, Val size: 397
Train loss первый батч: 0.27462705969810486
Val loss: 0.4436568319797516


In [5]:
from funcs.load_data import load_data
from funcs.nn_models import nn_predict, make_features, CNNPredictor
import torch
import pandas as pd
import numpy as np

data = load_data("./data")
train = data["SBER"][:pd.Timestamp("2025-01-03") - pd.Timedelta(days=1)].copy()

print("=== Проверка CNN через nn_predict ===")
preds = nn_predict("cnn", train, steps=1, lookback=20)
print(f"Прогноз CNN: {preds}")

# Диагностика данных
print("\n--- Диагностика ---")
X, y, scaler, df_feat = make_features(train, lookback=20)
print(f"Форма X: {X.shape}, y: {y.shape}")
print(f"X NaN: {np.isnan(X).any()}, y NaN: {np.isnan(y).any()}")

if len(X) > 9:
    val_size = max(1, int(0.2 * len(X)))
    X_train, X_val = X[:-val_size], X[-val_size:]
    y_train, y_val = y[:-val_size], y[-val_size:]
    print(f"Train size: {len(X_train)}, Val size: {len(X_val)}")

    model = CNNPredictor(input_size=X.shape[2])
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = torch.nn.MSELoss()
    X_t = torch.from_numpy(X_train).float()
    y_t = torch.from_numpy(y_train).float()
    optimizer.zero_grad()
    loss_train = criterion(model(X_t), y_t)
    print(f"Train loss (1-й батч): {loss_train.item()}")
    loss_train.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        loss_val = criterion(model(torch.from_numpy(X_val).float()), torch.from_numpy(y_val).float())
    print(f"Val loss: {loss_val.item()}")
else:
    print("Недостаточно окон")

=== Проверка CNN через nn_predict ===
Прогноз CNN: [256.60091845]

--- Диагностика ---
Форма X: (1986, 20, 19), y: (1986,)
X NaN: False, y NaN: False
Train size: 1589, Val size: 397
Train loss (1-й батч): 0.13650162518024445
Val loss: 0.20309625566005707


# Склеиваем метрики

In [2]:
stats_metrics = pd.read_excel('results/metrics/all_metrics.xlsx')
ml_metrics = pd.read_excel('results/metrics/ml_metrics.xlsx')
nn_metrics = pd.read_excel('results/metrics/nn_metrics.xlsx')

stats_metrics.shape, ml_metrics.shape, nn_metrics.shape

((36, 6), (24, 6), (24, 6))

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# ------------------------------
# 1. Загрузка и объединение данных
# ------------------------------
stats_metrics = pd.read_excel('results/metrics/all_metrics.xlsx')
ml_metrics = pd.read_excel('results/metrics/ml_metrics.xlsx')
nn_metrics = pd.read_excel('results/metrics/nn_metrics.xlsx')

stats_metrics['Тип'] = 'Статистические'
ml_metrics['Тип'] = 'ML'
nn_metrics['Тип'] = 'Нейросети'

df_all = pd.concat([stats_metrics, ml_metrics, nn_metrics], ignore_index=True)

# Убедимся, что горизонт – целое число (вдруг читается как float)
df_all['Горизонт'] = df_all['Горизонт'].astype(int)

In [6]:
# ------------------------------
# 2. Сводные таблицы по метрикам
# ------------------------------
# Для удобства создадим отдельные pivot для RMSE, MAE, MAPE с мультииндексом (Актив, Модель, Тип) и столбцами Горизонт
def pivot_metric(metric_name):
    pivot = df_all.pivot_table(
        index=['Актив', 'Модель', 'Тип'],
        columns='Горизонт',
        values=metric_name
    ).round(4)
    # Добавим среднее по всем горизонтам для быстрой оценки
    pivot['Среднее'] = pivot.mean(axis=1).round(4)
    return pivot

rmse_pivot = pivot_metric('RMSE')
mae_pivot  = pivot_metric('MAE')
mape_pivot = pivot_metric('MAPE')

# Сохраним сводные таблицы в один Excel с несколькими листами
os.makedirs('results/metrics', exist_ok=True)
with pd.ExcelWriter('results/metrics/combined_metrics_pivot.xlsx') as writer:
    rmse_pivot.to_excel(writer, sheet_name='RMSE')
    mae_pivot.to_excel(writer, sheet_name='MAE')
    mape_pivot.to_excel(writer, sheet_name='MAPE')

In [7]:
all_metrics = pd.read_excel('results/metrics/combined_metrics_pivot.xlsx')
all_metrics.shape

(28, 7)

In [8]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

stocks = ['SBER', 'YNDX']
crypto = ['BTC', 'ETH']

model_order = [
    'SMA_20', 'ETS', 'SARIMA',
    'RandomForest', 'CatBoost',
    'LSTM', 'CNN'
]

# Цвета для bar
color_map = {
    'SMA_20':       '#6baed6',   # светло-синий
    'ETS':          '#3182bd',   # синий
    'SARIMA':       '#08519c',   # тёмно-синий
    'RandomForest': '#74c476',   # светло-зелёный
    'CatBoost':     '#31a354',   # зелёный
    'LSTM':         '#fd8d3c',   # оранжевый
    'CNN':          '#e6550d'    # красно-оранжевый
}

# Настройки осей
tickvals = [1, 7, 21]
ticktext_short = ['1', '7', '21']
ticktext = ['H=1', 'H=7', 'H=21']

output_dir = 'results/metrics/plots'
os.makedirs(output_dir, exist_ok=True)

def plot_group(df_group, group_name):
    # Список активов в группе
    assets = sorted(df_group['Актив'].unique())

    # 1. Bar chart – вертикальные панели для RMSE/MAE, общая для MAPE
    for metric, y_label in [('RMSE', 'RMSE'), ('MAE', 'MAE'), ('MAPE', 'MAPE (%)')]:
        if metric in ['RMSE', 'MAE']:
            # Вертикальное размещение (активы друг под другом), оси Y независимы
            fig = px.bar(
                df_group,
                x='Горизонт',
                y=metric,
                color='Модель',
                facet_row='Актив',
                barmode='group',
                color_discrete_map=color_map,
                hover_data=['Тип'],
                title=f'{group_name} – {metric}',
                labels={'value': y_label}
            )
            # Отключаем общую ось Y – каждый ряд получает свой масштаб
            fig.update_yaxes(matches=None, title=None)
            # Можно добавить подпись к каждой оси Y
            for i, asset in enumerate(assets):
                fig.layout.annotations[i]['text'] = f'{asset} – {y_label}'
        else:
            # MAPE – горизонтальное размещение (или оставить как есть)
            fig = px.bar(
                df_group,
                x='Горизонт',
                y=metric,
                color='Модель',
                facet_col='Актив',
                barmode='group',
                color_discrete_map=color_map,
                hover_data=['Тип'],
                title=f'{group_name} – {metric}',
                labels={'value': y_label}
            )
        fig.update_xaxes(
            tickmode='array',
            tickvals=tickvals,
            ticktext=ticktext,
            tickangle=-90,
            type='category'
        )
        fig.update_layout(template='plotly_white', margin=dict(t=80))
        fig.update_layout(
            font=dict(size=14),                     # общий размер всех текстов
            title=dict(font=dict(size=20)),         # крупный заголовок
            legend=dict(
                font=dict(size=14),
                title=dict(text='Модель', font=dict(size=16))
            )
        )
        fig.update_xaxes(tickfont=dict(size=14), title_font=dict(size=16))
        fig.update_yaxes(tickfont=dict(size=14), title_font=dict(size=16))
        # fig.update_traces(width=0.6)

        # fig.write_html(os.path.join(output_dir, f'bar_{metric}_{group_name}.html'))
        fig.write_image(os.path.join(output_dir, f'bar_{metric}_{group_name}.png'), width=1200, height=600, scale=2)

    # 2. Line chart – аналогично: вертикальные панели для RMSE/MAE
    for metric in ['RMSE', 'MAE', 'MAPE']:
        if metric in ['RMSE', 'MAE']:
            fig = px.line(
                df_group,
                x='Горизонт',
                y=metric,
                color='Модель',
                line_dash='Тип',
                facet_row='Актив',
                markers=True,
                color_discrete_map=color_map,
                title=f'{group_name} – Динамика {metric}',
                labels={'value': metric}
            )
            fig.update_yaxes(matches=None, title=None)
            for i, asset in enumerate(assets):
                fig.layout.annotations[i]['text'] = f'{asset} – {metric}'
        else:
            fig = px.line(
                df_group,
                x='Горизонт',
                y=metric,
                color='Модель',
                line_dash='Тип',
                facet_col='Актив',
                markers=True,
                color_discrete_map=color_map,
                title=f'{group_name} – Динамика {metric}',
                labels={'value': metric}
            )
        fig.update_xaxes(
            tickmode='array',
            tickvals=tickvals,
            ticktext=ticktext,
            type='category'
        )
        fig.update_layout(template='plotly_white', hovermode='x unified')
        fig.write_html(os.path.join(output_dir, f'line_{metric}_{group_name}.html'))

    # 3. Heatmap
    # RMSE и MAE – два актива вертикально, независимые шкалы
    # MAPE – общая карта (или тоже вертикально, но без надобности)
    for metric in ['RMSE', 'MAE', 'MAPE']:
        if metric in ['RMSE', 'MAE'] and len(assets) == 2:
            # Создаём субплоты 2 строки x 1 столбец
            fig = make_subplots(
                rows=2, cols=1,
                subplot_titles=[f'{asset} – {metric}' for asset in assets],
                vertical_spacing=0.15
            )
            for idx, asset in enumerate(assets):
                df_asset = df_group[df_group['Актив'] == asset]
                pivot = df_asset.pivot_table(
                    index=['Модель', 'Тип'],
                    columns='Горизонт',
                    values=metric
                )
                pivot.columns = ['H=1', 'H=7', 'H=21']
                # Сортируем строки в нужном порядке
                ordered_rows = []
                for model in model_order:
                    for typ in ['Статистические', 'ML', 'Нейросети']:
                        if (model, typ) in pivot.index:
                            ordered_rows.append((model, typ))
                pivot = pivot.reindex(ordered_rows).dropna(how='all')
                y_labels = [f'{m} ({t})' for m, t in pivot.index]

                # Независимый zmin/zmax для этого актива
                zmin = pivot.min().min()
                zmax = pivot.max().max()

                heatmap = go.Heatmap(
                    z=pivot.values,
                    x=pivot.columns,
                    y=y_labels,
                    colorscale='RdYlGn_r',
                    zmin=zmin,
                    zmax=zmax,
                    text=np.round(pivot.values, 2),
                    texttemplate='%{text}',
                    colorbar=dict(title=metric, len=0.45, y=0.5 + (1-idx)*0.5),  # позиция цветовой полосы
                )
                fig.add_trace(heatmap, row=idx+1, col=1)
            
            fig.update_xaxes(tickmode='array', tickvals=[0,1,2], ticktext=['H=1','H=7','H=21'], row=1, col=1)
            fig.update_xaxes(tickmode='array', tickvals=[0,1,2], ticktext=['H=1','H=7','H=21'], row=2, col=1)
            fig.update_layout(
                template='plotly_white',
                title=f'{group_name} – Тепловая карта {metric}',
                height=800
            )
            fig.write_html(os.path.join(output_dir, f'heatmap_{metric}_{group_name}.html'))

        else:
            # MAPE или случай одного актива – общая тепловая карта
            pivot = df_group.pivot_table(
                index=['Актив', 'Модель', 'Тип'],
                columns='Горизонт',
                values=metric
            )
            pivot.columns = ['H=1', 'H=7', 'H=21']
            idx_tuples = []
            for asset in assets:
                for model in model_order:
                    for typ in ['Статистические', 'ML', 'Нейросети']:
                        tup = (asset, model, typ)
                        if tup in pivot.index:
                            idx_tuples.append(tup)
            pivot = pivot.reindex(idx_tuples).dropna(how='all')
            pivot.index = [f"{a} | {m} ({t})" for a, m, t in pivot.index]

            fig = px.imshow(
                pivot,
                text_auto='.2f',
                aspect='auto',
                color_continuous_scale='RdYlGn_r',
                title=f'{group_name} – Тепловая карта {metric}'
            )
            fig.update_layout(
                template='plotly_white',
                xaxis=dict(tickmode='array', tickvals=[0,1,2], ticktext=['H=1','H=7','H=21']),
            )
            fig.write_html(os.path.join(output_dir, f'heatmap_{metric}_{group_name}.html'))

# Запуск для акций и криптовалют
plot_group(df_all[df_all['Актив'].isin(stocks)], 'Акции')
plot_group(df_all[df_all['Актив'].isin(crypto)], 'Криптовалюты')

# Средний ранг RMSE по всем горизонтам для каждого актива и модели
rank_df = df_all.copy()
rank_df['Rank'] = rank_df.groupby(['Актив', 'Горизонт'])['RMSE'].rank(method='dense', ascending=True)
avg_rank = rank_df.groupby(['Актив', 'Модель', 'Тип'])['Rank'].mean().reset_index()
avg_rank = avg_rank.sort_values(['Актив', 'Rank'])

# График среднего ранга (горизонтальные столбцы)
for group, assets in [('Акции', stocks), ('Криптовалюты', crypto)]:
    df_plot = avg_rank[avg_rank['Актив'].isin(assets)]
    fig = px.bar(
        df_plot,
        x='Rank',
        y='Модель',
        color='Модель',
        orientation='h',
        facet_col='Актив',
        color_discrete_map=color_map,
        title=f'{group} – Средний ранг RMSE (меньше = точнее)',
        labels={'Rank': 'Средний ранг (1 – лучший)'}
    )
    fig.update_layout(template='plotly_white', showlegend=False)
    fig.write_html(os.path.join(output_dir, f'top_models_rank_{group}.html'))

# Таблица "Топ-3 модели по RMSE для каждого актива и горизонта"
top3 = df_all.sort_values(['Актив', 'Горизонт', 'RMSE']).groupby(['Актив', 'Горизонт']).head(3)
top3 = top3[['Актив', 'Горизонт', 'Модель', 'Тип', 'RMSE']].reset_index(drop=True)
top3.to_excel('results/metrics/top3_models_by_RMSE.xlsx', index=False)

print("Готово! Графики с цветами, heatmap и топ-модели сохранены.")

Готово! Графики с цветами, heatmap и топ-модели сохранены.


# Статистические модели на доходностях

In [2]:
import pandas as pd
import numpy as np
import os
from funcs.load_data import load_data, get_frequency
from funcs.stat_models import sma_predict, ets_predict, sarima_predict
from funcs.backtest import walk_forward_validation
from funcs.backtest import BacktesterReturns    # импорт из нового файла
from funcs.metrics import evaluate_metrics
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

TEST_START = "2025-01-01"
HORIZONS = (1, 7, 21)

data = load_data("./data")
returns_data = {}
prices_data = {}
for name, series in data.items():
    returns_data[name] = series.pct_change().dropna()
    prices_data[name] = series

base_result_dir = 'results_returns'
os.makedirs(base_result_dir, exist_ok=True)

all_results = {}
all_metrics = {}

for asset_name in returns_data:
    returns_series = returns_data[asset_name]
    prices = prices_data[asset_name]
    freq = get_frequency(prices)
    seasonal_periods = 5 if freq == 'B' else 7
    print(f"\n=== {asset_name} ===")

    models = {
        'SMA_20': lambda train, steps: sma_predict(train, steps, window=20),
        'ETS': lambda train, steps: ets_predict(train, steps, seasonal_periods=seasonal_periods),
        'SARIMA': lambda train, steps: sarima_predict(train, steps, seasonal_periods=seasonal_periods),
    }

    for model_name, model_func in models.items():
        print(f"\n  Модель: {model_name}")
        for h in HORIZONS:
            df_pred = walk_forward_validation(
                returns_series, model_func, test_start=TEST_START, horizon=h, logs=False
            )
            all_results[(asset_name, model_name, h)] = df_pred

            # evaluate_metrics с skip_mape=True
            metrics = evaluate_metrics(df_pred, skip_mape=True)
            all_metrics[(asset_name, model_name, h)] = metrics
            print(f"    H={h}: RMSE={metrics['RMSE']:.6f}, MAE={metrics['MAE']:.6f}")

            # Бэктест
            bt = BacktesterReturns(
                predictions_df=df_pred,
                prices=prices,
                initial_capital=100_000,
                threshold=0.005,
                commission=0.001,
            )
            results_bt, trades_bt, fig_bt = bt.run()

            result_subdir = os.path.join(base_result_dir, f'1_{model_name}')
            for sub in ['predictions', 'backtest', 'plots']:
                os.makedirs(os.path.join(result_subdir, sub), exist_ok=True)

            df_pred.to_excel(os.path.join(result_subdir, 'predictions', f'{asset_name}_preds_{model_name}_{h}.xlsx'))
            results_bt.to_excel(os.path.join(result_subdir, 'backtest', f'{asset_name}_backtest_{model_name}_{h}.xlsx'))
            trades_bt.to_excel(os.path.join(result_subdir, 'backtest', f'{asset_name}_trades_{model_name}_{h}.xlsx'))
            fig_bt.write_html(os.path.join(result_subdir, 'plots', f'{asset_name}_{model_name}_{h}.html'))

# Сохранение метрик
metrics_rows = []
for (asset, model, h), m in all_metrics.items():
    metrics_rows.append({
        'Актив': asset,
        'Модель': model,
        'Горизонт': h,
        'RMSE': m['RMSE'],
        'MAE': m['MAE'],
    })
metrics_df = pd.DataFrame(metrics_rows)
os.makedirs(os.path.join(base_result_dir, 'metrics'), exist_ok=True)
metrics_df.to_excel(os.path.join(base_result_dir, 'metrics', 'stat_returns_metrics.xlsx'), index=False)
print("\nСводные метрики (RMSE, MAE) сохранены.")


=== SBER ===

  Модель: SMA_20


  H=1: 100%|██████████| 335/335 [00:00<00:00, 2865.81window/s]

    H=1: RMSE=0.013209, MAE=0.009091



  H=7: 48window [00:00, 765.21window/s]          


    H=7: RMSE=0.013209, MAE=0.009101


...

    H=1: RMSE=0.039142, MAE=0.027428


  H=7: 70window [07:04,  6.06s/window]                    


    H=7: RMSE=0.039065, MAE=0.027238


  H=21: 24window [02:27,  6.17s/window]                    

    H=21: RMSE=0.039068, MAE=0.027276

Сводные метрики (RMSE, MAE) сохранены.


# Классические ML модели на доходностях

In [ ]:
import pandas as pd
import numpy as np
import os
from funcs.load_data import load_data
from funcs.backtest import walk_forward_validation
from funcs.backtest import BacktesterReturns
from funcs.features_create import create_features_returns
from funcs.metrics import evaluate_metrics
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

TEST_START = "2025-01-01"
HORIZONS = (1, 7, 21)

# Загрузка цен
data = load_data("./data")
print("Активы:", list(data.keys()))

# Готовим цены (для бэктестера) и доходности (только для определения частоты в walk_forward)
prices_data = data
returns_data = {name: series.pct_change().dropna() for name, series in data.items()}

ML_MODELS = {
    'RandomForest': {
        'model_class': RandomForestRegressor,
        'model_params': {'n_estimators': 200, 'max_depth': 10, 'random_state': 42, 'n_jobs': -1}
    },
    'CatBoost': {
        'model_class': CatBoostRegressor,
        'model_params': {'iterations': 200, 'depth': 6, 'random_seed': 42, 'verbose': False, 'thread_count': -1}
    }
}

base_result_dir = 'results_returns'
os.makedirs(base_result_dir, exist_ok=True)

all_results = {}
all_metrics = {}

for asset_name in data:
    close = data[asset_name]
    returns_series = returns_data[asset_name]
    prices = close
    print(f"\n=== {asset_name} ===")

    for model_name, cfg in ML_MODELS.items():
        print(f"  Модель: {model_name}")
        for h in HORIZONS:
            # Генерируем признаки на основе ЦЕН (close) и горизонт h
            X, y = create_features_returns(close, horizon=h)

            # Синхронизируем индекс X, y с returns_series
            common_idx = returns_series.index.intersection(X.index).intersection(y.index)
            X = X.loc[common_idx]
            y = y.loc[common_idx]
            series_for_wf = returns_series[common_idx]   # только для дат и частоты

            if len(common_idx) < 20:
                print(f"    Пропуск: недостаточно общих наблюдений для {asset_name} H={h}")
                continue

            ml_config = {
                'X': X,
                'y': y,
                'model_class': cfg['model_class'],
                'model_params': cfg['model_params']
            }

            # Walk-forward validation (ML-режим)
            df_pred = walk_forward_validation(
                series_for_wf,
                ml_config=ml_config,
                test_start=TEST_START,
                horizon=h,
                logs=False
            )
            all_results[(asset_name, model_name, h)] = df_pred

            # Метрики (без MAPE)
            metrics = evaluate_metrics(df_pred, skip_mape=True)
            all_metrics[(asset_name, model_name, h)] = metrics
            print(f"    H={h}: RMSE={metrics['RMSE']:.6f}, MAE={metrics['MAE']:.6f}")

            # Бэктест через BacktesterReturns
            bt = BacktesterReturns(
                predictions_df=df_pred,
                prices=prices,
                initial_capital=100_000,
                threshold=0.005,
                commission=0.001,
            )
            results_bt, trades_bt, fig_bt = bt.run()

            # Сохранение результатов
            result_subdir = os.path.join(base_result_dir, f'2_{model_name}')
            for sub in ['predictions', 'backtest', 'plots']:
                os.makedirs(os.path.join(result_subdir, sub), exist_ok=True)

            df_pred.to_excel(os.path.join(result_subdir, 'predictions', f'{asset_name}_preds_{model_name}_{h}.xlsx'))
            results_bt.to_excel(os.path.join(result_subdir, 'backtest', f'{asset_name}_backtest_{model_name}_{h}.xlsx'))
            trades_bt.to_excel(os.path.join(result_subdir, 'backtest', f'{asset_name}_trades_{model_name}_{h}.xlsx'))
            fig_bt.write_html(os.path.join(result_subdir, 'plots', f'{asset_name}_{model_name}_{h}.html'))

# Сохраняем сводную таблицу метрик
metrics_rows = []
for (asset, model, h), m in all_metrics.items():
    metrics_rows.append({
        'Актив': asset,
        'Модель': model,
        'Горизонт': h,
        'RMSE': m['RMSE'],
        'MAE': m['MAE'],
    })
metrics_df = pd.DataFrame(metrics_rows)
os.makedirs(os.path.join(base_result_dir, 'metrics'), exist_ok=True)
metrics_df.to_excel(os.path.join(base_result_dir, 'metrics', 'ml_returns_metrics.xlsx'), index=False)
print("\nИтоговая таблица ML на доходностях:")
print(metrics_df)

Активы: ['SBER', 'YNDX', 'BTC', 'ETH']

=== SBER ===
  Модель: RandomForest


  H=1: 100%|██████████| 334/334 [01:20<00:00,  4.15window/s]


    H=1: RMSE=0.012668, MAE=0.008664


  H=7: 47window [00:11,  4.19window/s]                    


    H=7: RMSE=0.032429, MAE=0.024416


...

    H=1: RMSE=0.041260, MAE=0.029943


  H=7: 69window [00:18,  3.65window/s]                    


    H=7: RMSE=0.104753, MAE=0.079896


  H=21: 23window [00:06,  3.70window/s]                    

    H=21: RMSE=0.206184, MAE=0.155754

Итоговая таблица ML на доходностях:
   Актив        Модель  Горизонт      RMSE       MAE
0   SBER  RandomForest         1  0.012668  0.008664
1   SBER  RandomForest         7  0.032429  0.024416
2   SBER  RandomForest        21  0.052281  0.039691
3   SBER      CatBoost         1  0.014338  0.009720
4   SBER      CatBoost         7  0.037989  0.029679
5   SBER      CatBoost        21  0.068238  0.052651
6   YNDX  RandomForest         1  0.017118  0.013067
7   YNDX  RandomForest         7  0.041540  0.033322
8   YNDX  RandomForest        21  0.072296  0.058084
9   YNDX      CatBoost         1  0.017944  0.013879
10  YNDX      CatBoost         7  0.046511  0.037502
11  YNDX      CatBoost        21  0.080547  0.065713
12   BTC  RandomForest         1  0.023623  0.016697
13   BTC  RandomForest         7  0.060822  0.046985
14   BTC  RandomForest        21  0.127889  0.098058
15   BTC      CatBoost         1  0.025462  0.018200
16   BTC      CatBoost  

In [ ]:
# Сохраняем сводную таблицу метрик
metrics_rows = []
for (asset, model, h), m in all_metrics.items():
    metrics_rows.append({
        'Актив': asset,
        'Модель': model,
        'Горизонт': h,
        'RMSE': m['RMSE'],
        'MAE': m['MAE'],
    })
metrics_df = pd.DataFrame(metrics_rows)
os.makedirs(os.path.join(base_result_dir, 'metrics'), exist_ok=True)
metrics_df.to_excel(os.path.join(base_result_dir, 'metrics', 'ml_returns_metrics.xlsx'), index=False)
print("\nИтоговая таблица ML на доходностях:")
print(metrics_df)

In [2]:
import pandas as pd
import numpy as np
import os
from funcs.load_data import load_data
from funcs.backtest import walk_forward_validation
from funcs.backtest import BacktesterReturns
from funcs.features_create import create_features_returns
from funcs.metrics import evaluate_metrics
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

TEST_START = "2025-01-01"
HORIZONS = (1,)

# Загрузка цен
data = load_data("./data")
print("Активы:", list(data.keys()))

# Готовим цены (для бэктестера) и доходности (только для определения частоты в walk_forward)
prices_data = data
returns_data = {name: series.pct_change().dropna() for name, series in data.items()}

ML_MODELS = {
    # 'RandomForest': {
    #     'model_class': RandomForestRegressor,
    #     'model_params': {'n_estimators': 200, 'max_depth': 10, 'random_state': 42, 'n_jobs': -1}
    # },
    'CatBoost': {
        'model_class': CatBoostRegressor,
        'model_params': {'iterations': 200, 'depth': 6, 'random_seed': 42, 'verbose': False, 'thread_count': -1}
    }
}

base_result_dir = 'results_returns'
os.makedirs(base_result_dir, exist_ok=True)

all_results = {}
all_metrics = {}

for asset_name in data:
    close = data[asset_name]
    returns_series = returns_data[asset_name]
    prices = close
    print(f"\n=== {asset_name} ===")

    for model_name, cfg in ML_MODELS.items():
        print(f"  Модель: {model_name}")
        for h in HORIZONS:
            # Генерируем признаки на основе ЦЕН (close) и горизонт h
            X, y = create_features_returns(close, horizon=h)

            # Синхронизируем индекс X, y с returns_series
            common_idx = returns_series.index.intersection(X.index).intersection(y.index)
            X = X.loc[common_idx]
            y = y.loc[common_idx]
            series_for_wf = returns_series[common_idx]   # только для дат и частоты

            if len(common_idx) < 20:
                print(f"    Пропуск: недостаточно общих наблюдений для {asset_name} H={h}")
                continue

            ml_config = {
                'X': X,
                'y': y,
                'model_class': cfg['model_class'],
                'model_params': cfg['model_params']
            }

            # Walk-forward validation (ML-режим)
            df_pred = walk_forward_validation(
                series_for_wf,
                ml_config=ml_config,
                test_start=TEST_START,
                horizon=h,
                logs=False
            )
            all_results[(asset_name, model_name, h)] = df_pred

            # Метрики (без MAPE)
            metrics = evaluate_metrics(df_pred, skip_mape=True)
            all_metrics[(asset_name, model_name, h)] = metrics
            print(f"    H={h}: RMSE={metrics['RMSE']:.6f}, MAE={metrics['MAE']:.6f}")

            # Бэктест через BacktesterReturns
            bt = BacktesterReturns(
                predictions_df=df_pred,
                prices=prices,
                initial_capital=100_000,
                threshold=0.005,
                commission=0.001,
            )
            results_bt, trades_bt, fig_bt = bt.run()

            # Сохранение результатов
            result_subdir = os.path.join(base_result_dir, f'2_{model_name}')
            for sub in ['predictions', 'backtest', 'plots']:
                os.makedirs(os.path.join(result_subdir, sub), exist_ok=True)

            df_pred.to_excel(os.path.join(result_subdir, 'predictions', f'{asset_name}_preds_{model_name}_{h}.xlsx'))
            results_bt.to_excel(os.path.join(result_subdir, 'backtest', f'{asset_name}_backtest_{model_name}_{h}.xlsx'))
            trades_bt.to_excel(os.path.join(result_subdir, 'backtest', f'{asset_name}_trades_{model_name}_{h}.xlsx'))
            fig_bt.write_html(os.path.join(result_subdir, 'plots', f'{asset_name}_{model_name}_{h}.html'))
            fig_bt.write_image(os.path.join(result_subdir, 'plots', f'{asset_name}_{model_name}_{h}.png'), width=1600, height=1400, scale=2)


Активы: ['SBER', 'YNDX', 'BTC', 'ETH']

=== SBER ===
  Модель: CatBoost


  H=1: 100%|██████████| 334/334 [01:22<00:00,  4.03window/s]


    H=1: RMSE=0.014338, MAE=0.009720

=== YNDX ===
  Модель: CatBoost


  H=1: 100%|██████████| 334/334 [01:23<00:00,  3.98window/s]


    H=1: RMSE=0.017944, MAE=0.013879

=== BTC ===
  Модель: CatBoost


  H=1: 100%|██████████| 484/484 [02:12<00:00,  3.65window/s]


    H=1: RMSE=0.025462, MAE=0.018200

=== ETH ===
  Модель: CatBoost


  H=1: 100%|██████████| 484/484 [02:10<00:00,  3.71window/s]


    H=1: RMSE=0.041260, MAE=0.029943


# Нейросетевые модели на доходностях

In [ ]:
import pandas as pd
import numpy as np
import os
from funcs.load_data import load_data
from funcs.backtest import walk_forward_validation
from funcs.backtest import BacktesterReturns
from funcs.nn_models import nn_predict_returns
from funcs.metrics import evaluate_metrics
import warnings
warnings.filterwarnings('ignore')

TEST_START = "2025-01-01"
HORIZONS = (1, 7, 21)
# HORIZONS = (21,)

# Загрузка цен
data = load_data("./data")
print("Активы:", list(data.keys()))

# Доходности и цены
returns_data = {name: series.pct_change().dropna() for name, series in data.items()}
prices_data = data

NN_MODELS = {
    'LSTM': lambda train, steps: nn_predict_returns('lstm', train, steps, lookback=20),
    'CNN':  lambda train, steps: nn_predict_returns('cnn', train, steps, lookback=20),
}

base_result_dir = 'results_returns'
os.makedirs(base_result_dir, exist_ok=True)

all_results = {}
all_metrics = {}

for asset_name in data:
    returns_series = returns_data[asset_name]
    prices = prices_data[asset_name]
    print(f"\n=== {asset_name} ===")

    for model_name, model_func in NN_MODELS.items():
        print(f"  Модель: {model_name}")
        for h in HORIZONS:
            # walk_forward_validation использует ряд доходностей (для дат и частоты)
            df_pred = walk_forward_validation(
                returns_series,
                model_func,
                test_start=TEST_START,
                horizon=h,
                logs=False
            )
            all_results[(asset_name, model_name, h)] = df_pred

            # Метрики (MAPE пропускаем)
            metrics = evaluate_metrics(df_pred, skip_mape=True)
            all_metrics[(asset_name, model_name, h)] = metrics
            print(f"    H={h}: RMSE={metrics['RMSE']:.6f}, MAE={metrics['MAE']:.6f}")

            # Бэктест
            bt = BacktesterReturns(
                predictions_df=df_pred,
                prices=prices,
                initial_capital=100_000,
                threshold=0.005,
                commission=0.001,
            )
            results_bt, trades_bt, fig_bt = bt.run()

            # Сохранение
            result_subdir = os.path.join(base_result_dir, f'3_{model_name}')
            for sub in ['predictions', 'backtest', 'plots']:
                os.makedirs(os.path.join(result_subdir, sub), exist_ok=True)

            df_pred.to_excel(os.path.join(result_subdir, 'predictions', f'{asset_name}_preds_{model_name}_{h}.xlsx'))
            results_bt.to_excel(os.path.join(result_subdir, 'backtest', f'{asset_name}_backtest_{model_name}_{h}.xlsx'))
            trades_bt.to_excel(os.path.join(result_subdir, 'backtest', f'{asset_name}_trades_{model_name}_{h}.xlsx'))
            fig_bt.write_html(os.path.join(result_subdir, 'plots', f'{asset_name}_{model_name}_{h}.html'))


Активы: ['SBER', 'YNDX', 'BTC', 'ETH']

=== SBER ===
  Модель: LSTM


  H=1: 100%|██████████| 335/335 [35:37<00:00,  6.38s/window]


    H=1: RMSE=0.013741, MAE=0.009795


  H=7: 48window [04:36,  5.76s/window]                    


    H=7: RMSE=0.013986, MAE=0.009889


...

    H=1: RMSE=0.042135, MAE=0.030650


  H=7: 70window [04:18,  3.69s/window]                    


    H=7: RMSE=0.040983, MAE=0.028679


  H=21: 24window [01:32,  3.84s/window]                    


    H=21: RMSE=0.040651, MAE=0.029606


In [3]:
# Сохранение сводных метрик
metrics_rows = []
for (asset, model, h), m in all_metrics.items():
    metrics_rows.append({
        'Актив': asset,
        'Модель': model,
        'Горизонт': h,
        'RMSE': m['RMSE'],
        'MAE': m['MAE'],
    })
metrics_df = pd.DataFrame(metrics_rows)
os.makedirs(os.path.join(base_result_dir, 'metrics'), exist_ok=True)
metrics_df.to_excel(os.path.join(base_result_dir, 'metrics', 'nn_returns_metrics.xlsx'), index=False)
print("\nИтоговая таблица метрик нейросетей на доходностях:")
print(metrics_df)


Итоговая таблица метрик нейросетей на доходностях:
   Актив Модель  Горизонт      RMSE       MAE
0   SBER   LSTM         1  0.013741  0.009795
1   SBER   LSTM         7  0.013986  0.009889
2   SBER   LSTM        21  0.014321  0.010212
3   SBER    CNN         1  0.015996  0.011849
4   SBER    CNN         7  0.016330  0.011906
5   SBER    CNN        21  0.017958  0.013006
6   YNDX   LSTM         1  0.017754  0.013396
7   YNDX   LSTM         7  0.018373  0.013732
8   YNDX   LSTM        21  0.017826  0.013629
9   YNDX    CNN         1  0.020305  0.015650
10  YNDX    CNN         7  0.018892  0.014374
11  YNDX    CNN        21  0.021330  0.016947
12   BTC   LSTM         1  0.025119  0.018133
13   BTC   LSTM         7  0.025381  0.018715
14   BTC   LSTM        21  0.025184  0.018388
15   BTC    CNN         1  0.024930  0.018146
16   BTC    CNN         7  0.026502  0.019425
17   BTC    CNN        21  0.027542  0.020031
18   ETH   LSTM         1  0.040152  0.028426
19   ETH   LSTM         7  0

# Склеиваем метрики для доходностей

In [1]:
import pandas as pd
import numpy as np
import os

# Корневые папки с результатами
ROOT_PRICES = 'results'
ROOT_RETURNS = 'results_returns'

# Горизонты и активы
HORIZONS = [1, 7, 21]
ASSETS = ['SBER', 'YNDX', 'BTC', 'ETH']

def compute_sharpe(returns_series, periods=252):
    """Годовой коэффициент Шарпа по дневным доходностям"""
    # возвращает NaN, если std = 0 или мало наблюдений
    if returns_series.std() == 0 or len(returns_series) < 2:
        return np.nan
    return returns_series.mean() / returns_series.std() * np.sqrt(periods)

def compute_max_drawdown(equity_series):
    """Максимальная просадка в процентах (отрицательное число)"""
    cummax = equity_series.cummax()
    drawdown = (equity_series - cummax) / cummax
    return drawdown.min()

def aggregate_from_folder(base_dir, model_prefixes):
    """
    base_dir: 'results' или 'results_returns'
    model_prefixes: список папок моделей (например, '1_SMA_20', '2_CatBoost', '3_LSTM'...)
    """
    records = []
    for prefix in model_prefixes:
        model_name = prefix.split('_', 1)[1] if '_' in prefix else prefix  # SMA_20, CatBoost ...
        backtest_dir = os.path.join(base_dir, prefix, 'backtest')
        if not os.path.isdir(backtest_dir):
            continue
        for asset in ASSETS:
            for h in HORIZONS:
                # файлы бэктеста и журнала сделок
                bt_file = os.path.join(backtest_dir, f'{asset}_backtest_{model_name}_{h}.xlsx')
                trades_file = os.path.join(backtest_dir, f'{asset}_trades_{model_name}_{h}.xlsx')
                if not os.path.isfile(bt_file):
                    continue
                try:
                    results_df = pd.read_excel(bt_file, index_col=0)
                    trades_df = pd.read_excel(trades_file) if os.path.isfile(trades_file) else pd.DataFrame()
                except Exception as e:
                    print(f'Ошибка чтения {bt_file}: {e}')
                    continue

                # Итоговый капитал — последнее значение equity
                if 'equity' not in results_df.columns or results_df.empty:
                    continue
                final_equity = results_df['equity'].iloc[-1]

                # Шарп — по столбцу strategy_return, если он есть
                sharpe = np.nan
                if 'strategy_return' in results_df.columns:
                    rets = results_df['strategy_return'].dropna()
                    sharpe = compute_sharpe(rets)

                # Максимальная просадка
                max_dd = compute_max_drawdown(results_df['equity'])

                # Количество сделок и % прибыльных
                num_trades = len(trades_df)
                if num_trades > 0 and 'strategy_return' in trades_df.columns:
                    win_rate = (trades_df['strategy_return'] > 0).mean()
                else:
                    win_rate = np.nan

                records.append({
                    'Актив': asset,
                    'Модель': model_name,
                    'Горизонт': h,
                    'Итоговый капитал': final_equity,
                    'Шарп': sharpe,
                    'Макс. просадка': max_dd,
                    'Сделок': num_trades,
                    'Прибыльных, %': win_rate
                })
    return pd.DataFrame(records)

# Определяем папки моделей для цен
if os.path.isdir(ROOT_PRICES):
    price_folders = [d for d in os.listdir(ROOT_PRICES) if os.path.isdir(os.path.join(ROOT_PRICES, d))]
else:
    price_folders = []
print('Папки моделей (цены):', price_folders)

# Определяем папки моделей для доходностей
if os.path.isdir(ROOT_RETURNS):
    returns_folders = [d for d in os.listdir(ROOT_RETURNS) if os.path.isdir(os.path.join(ROOT_RETURNS, d))]
else:
    returns_folders = []
print('Папки моделей (доходности):', returns_folders)

# Агрегируем
df_prices = aggregate_from_folder(ROOT_PRICES, price_folders)
df_returns = aggregate_from_folder(ROOT_RETURNS, returns_folders)

# Сохраняем
df_prices.to_excel('summary_prices.xlsx', index=False)
df_returns.to_excel('summary_returns.xlsx', index=False)

print('Готово! Сохранены summary_prices.xlsx и summary_returns.xlsx')

Папки моделей (цены): ['1_ETS', '1_SARIMA', '1_SMA_20', '2_CatBoost', '2_RandomForest', '3_cnn', '3_LSTM', 'metrics']
Папки моделей (доходности): ['1_ETS', '1_SARIMA', '1_SMA_20', '2_CatBoost', '2_RandomForest', '3_CNN', '3_LSTM', 'metrics']
Готово! Сохранены summary_prices.xlsx и summary_returns.xlsx


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# ------------------------------
# 1. Загрузка и объединение данных
# ------------------------------
stats_metrics = pd.read_excel('results_returns/metrics/all_metrics.xlsx')
ml_metrics = pd.read_excel('results_returns/metrics/ml_metrics.xlsx')
nn_metrics = pd.read_excel('results_returns/metrics/nn_metrics.xlsx')

stats_metrics['Тип'] = 'Статистические'
ml_metrics['Тип'] = 'ML'
nn_metrics['Тип'] = 'Нейросети'

df_all = pd.concat([stats_metrics, ml_metrics, nn_metrics], ignore_index=True)

# Убедимся, что горизонт – целое число (вдруг читается как float)
df_all['Горизонт'] = df_all['Горизонт'].astype(int)

In [2]:
# ------------------------------
# 2. Сводные таблицы по метрикам
# ------------------------------
# Для удобства создадим отдельные pivot для RMSE, MAE, MAPE с мультииндексом (Актив, Модель, Тип) и столбцами Горизонт
def pivot_metric(metric_name):
    pivot = df_all.pivot_table(
        index=['Актив', 'Модель', 'Тип'],
        columns='Горизонт',
        values=metric_name
    ).round(4)
    # Добавим среднее по всем горизонтам для быстрой оценки
    pivot['Среднее'] = pivot.mean(axis=1).round(4)
    return pivot

rmse_pivot = pivot_metric('RMSE')
mae_pivot  = pivot_metric('MAE')

# Сохраним сводные таблицы в один Excel с несколькими листами
os.makedirs('results_returns/metrics', exist_ok=True)
with pd.ExcelWriter('results_returns/metrics/combined_metrics_pivot.xlsx') as writer:
    rmse_pivot.to_excel(writer, sheet_name='RMSE')
    mae_pivot.to_excel(writer, sheet_name='MAE')


In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# ------------------------------
# 1. Загрузка и объединение данных
# ------------------------------
stats_metrics = pd.read_excel('results_returns/metrics/all_metrics.xlsx')
ml_metrics = pd.read_excel('results_returns/metrics/ml_metrics.xlsx')
nn_metrics = pd.read_excel('results_returns/metrics/nn_metrics.xlsx')

stats_metrics['Тип'] = 'Статистические'
ml_metrics['Тип'] = 'ML'
nn_metrics['Тип'] = 'Нейросети'

df_all = pd.concat([stats_metrics, ml_metrics, nn_metrics], ignore_index=True)

# Убедимся, что горизонт – целое число (вдруг читается как float)
df_all['Горизонт'] = df_all['Горизонт'].astype(int)
# ------------------------------
# 2. Сводные таблицы по метрикам
# ------------------------------
# Для удобства создадим отдельные pivot для RMSE, MAE, MAPE с мультииндексом (Актив, Модель, Тип) и столбцами Горизонт
def pivot_metric(metric_name):
    pivot = df_all.pivot_table(
        index=['Актив', 'Модель', 'Тип'],
        columns='Горизонт',
        values=metric_name
    ).round(4)
    # Добавим среднее по всем горизонтам для быстрой оценки
    pivot['Среднее'] = pivot.mean(axis=1).round(4)
    return pivot

rmse_pivot = pivot_metric('RMSE')
mae_pivot  = pivot_metric('MAE')

# Сохраним сводные таблицы в один Excel с несколькими листами
os.makedirs('results_returns/metrics', exist_ok=True)
with pd.ExcelWriter('results_returns/metrics/combined_metrics_pivot.xlsx') as writer:
    rmse_pivot.to_excel(writer, sheet_name='RMSE')
    mae_pivot.to_excel(writer, sheet_name='MAE')
all_metrics = pd.read_excel('results_returns/metrics/combined_metrics_pivot.xlsx')
all_metrics.shape
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

stocks = ['SBER', 'YNDX']
crypto = ['BTC', 'ETH']

model_order = [
    'SMA_20', 'ETS', 'SARIMA',
    'RandomForest', 'CatBoost',
    'LSTM', 'CNN'
]

# Цвета для bar
color_map = {
    'SMA_20':       '#6baed6',   # светло-синий
    'ETS':          '#3182bd',   # синий
    'SARIMA':       '#08519c',   # тёмно-синий
    'RandomForest': '#74c476',   # светло-зелёный
    'CatBoost':     '#31a354',   # зелёный
    'LSTM':         '#fd8d3c',   # оранжевый
    'CNN':          '#e6550d'    # красно-оранжевый
}

# Настройки осей
tickvals = [1, 7, 21]
ticktext_short = ['1', '7', '21']
ticktext = ['H=1', 'H=7', 'H=21']

output_dir = 'results_returns/metrics/plots'
os.makedirs(output_dir, exist_ok=True)

def plot_group(df_group, group_name):
    # Список активов в группе
    assets = sorted(df_group['Актив'].unique())

    # 1. Bar chart – вертикальные панели для RMSE/MAE, общая для MAPE
    for metric, y_label in [('RMSE', 'RMSE'), ('MAE', 'MAE')]:
        if metric in ['RMSE', 'MAE']:
            # Вертикальное размещение (активы друг под другом), оси Y независимы
            fig = px.bar(
                df_group,
                x='Горизонт',
                y=metric,
                color='Модель',
                facet_row='Актив',
                barmode='group',
                color_discrete_map=color_map,
                hover_data=['Тип'],
                title=f'{group_name} – {metric}',
                labels={'value': y_label}
            )
            # Отключаем общую ось Y – каждый ряд получает свой масштаб
            fig.update_yaxes(matches=None, title=None)
            # Можно добавить подпись к каждой оси Y
            for i, asset in enumerate(assets):
                fig.layout.annotations[i]['text'] = f'{asset} – {y_label}'
        else:
            # MAPE – горизонтальное размещение (или оставить как есть)
            fig = px.bar(
                df_group,
                x='Горизонт',
                y=metric,
                color='Модель',
                facet_col='Актив',
                barmode='group',
                color_discrete_map=color_map,
                hover_data=['Тип'],
                title=f'{group_name} – {metric}',
                labels={'value': y_label}
            )
        fig.update_xaxes(
            tickmode='array',
            tickvals=tickvals,
            ticktext=ticktext,
            tickangle=-90,
            type='category'
        )
        fig.update_layout(template='plotly_white', margin=dict(t=80))
        fig.update_layout(
            font=dict(size=14),                     # общий размер всех текстов
            title=dict(font=dict(size=20)),         # крупный заголовок
            legend=dict(
                font=dict(size=14),
                title=dict(text='Модель', font=dict(size=16))
            )
        )
        fig.update_xaxes(tickfont=dict(size=14), title_font=dict(size=16))
        fig.update_yaxes(tickfont=dict(size=14), title_font=dict(size=16))
        # fig.update_traces(width=0.6)

        # fig.write_html(os.path.join(output_dir, f'bar_{metric}_{group_name}.html'))
        fig.write_image(os.path.join(output_dir, f'bar_{metric}_{group_name}.png'), width=1200, height=600, scale=2)

    # 2. Line chart – аналогично: вертикальные панели для RMSE/MAE
    for metric in ['RMSE', 'MAE']:
        if metric in ['RMSE', 'MAE']:
            fig = px.line(
                df_group,
                x='Горизонт',
                y=metric,
                color='Модель',
                line_dash='Тип',
                facet_row='Актив',
                markers=True,
                color_discrete_map=color_map,
                title=f'{group_name} – Динамика {metric}',
                labels={'value': metric}
            )
            fig.update_yaxes(matches=None, title=None)
            for i, asset in enumerate(assets):
                fig.layout.annotations[i]['text'] = f'{asset} – {metric}'
        else:
            fig = px.line(
                df_group,
                x='Горизонт',
                y=metric,
                color='Модель',
                line_dash='Тип',
                facet_col='Актив',
                markers=True,
                color_discrete_map=color_map,
                title=f'{group_name} – Динамика {metric}',
                labels={'value': metric}
            )
        fig.update_xaxes(
            tickmode='array',
            tickvals=tickvals,
            ticktext=ticktext,
            type='category'
        )
        fig.update_layout(template='plotly_white', hovermode='x unified')
        fig.write_html(os.path.join(output_dir, f'line_{metric}_{group_name}.html'))

    # 3. Heatmap
    # RMSE и MAE – два актива вертикально, независимые шкалы
    # MAPE – общая карта (или тоже вертикально, но без надобности)
    for metric in ['RMSE', 'MAE']:
        if metric in ['RMSE', 'MAE'] and len(assets) == 2:
            # Создаём субплоты 2 строки x 1 столбец
            fig = make_subplots(
                rows=2, cols=1,
                subplot_titles=[f'{asset} – {metric}' for asset in assets],
                vertical_spacing=0.15
            )
            for idx, asset in enumerate(assets):
                df_asset = df_group[df_group['Актив'] == asset]
                pivot = df_asset.pivot_table(
                    index=['Модель', 'Тип'],
                    columns='Горизонт',
                    values=metric
                )
                pivot.columns = ['H=1', 'H=7', 'H=21']
                # Сортируем строки в нужном порядке
                ordered_rows = []
                for model in model_order:
                    for typ in ['Статистические', 'ML', 'Нейросети']:
                        if (model, typ) in pivot.index:
                            ordered_rows.append((model, typ))
                pivot = pivot.reindex(ordered_rows).dropna(how='all')
                y_labels = [f'{m} ({t})' for m, t in pivot.index]

                # Независимый zmin/zmax для этого актива
                zmin = pivot.min().min()
                zmax = pivot.max().max()

                heatmap = go.Heatmap(
                    z=pivot.values,
                    x=pivot.columns,
                    y=y_labels,
                    colorscale='RdYlGn_r',
                    zmin=zmin,
                    zmax=zmax,
                    text=np.round(pivot.values, 2),
                    texttemplate='%{text}',
                    colorbar=dict(title=metric, len=0.45, y=0.5 + (1-idx)*0.5),  # позиция цветовой полосы
                )
                fig.add_trace(heatmap, row=idx+1, col=1)
            
            fig.update_xaxes(tickmode='array', tickvals=[0,1,2], ticktext=['H=1','H=7','H=21'], row=1, col=1)
            fig.update_xaxes(tickmode='array', tickvals=[0,1,2], ticktext=['H=1','H=7','H=21'], row=2, col=1)
            fig.update_layout(
                template='plotly_white',
                title=f'{group_name} – Тепловая карта {metric}',
                height=800
            )
            fig.write_html(os.path.join(output_dir, f'heatmap_{metric}_{group_name}.html'))

        else:
            # MAPE или случай одного актива – общая тепловая карта
            pivot = df_group.pivot_table(
                index=['Актив', 'Модель', 'Тип'],
                columns='Горизонт',
                values=metric
            )
            pivot.columns = ['H=1', 'H=7', 'H=21']
            idx_tuples = []
            for asset in assets:
                for model in model_order:
                    for typ in ['Статистические', 'ML', 'Нейросети']:
                        tup = (asset, model, typ)
                        if tup in pivot.index:
                            idx_tuples.append(tup)
            pivot = pivot.reindex(idx_tuples).dropna(how='all')
            pivot.index = [f"{a} | {m} ({t})" for a, m, t in pivot.index]

            fig = px.imshow(
                pivot,
                text_auto='.2f',
                aspect='auto',
                color_continuous_scale='RdYlGn_r',
                title=f'{group_name} – Тепловая карта {metric}'
            )
            fig.update_layout(
                template='plotly_white',
                xaxis=dict(tickmode='array', tickvals=[0,1,2], ticktext=['H=1','H=7','H=21']),
            )
            fig.write_html(os.path.join(output_dir, f'heatmap_{metric}_{group_name}.html'))

# Запуск для акций и криптовалют
plot_group(df_all[df_all['Актив'].isin(stocks)], 'Акции')
plot_group(df_all[df_all['Актив'].isin(crypto)], 'Криптовалюты')

# Средний ранг RMSE по всем горизонтам для каждого актива и модели
rank_df = df_all.copy()
rank_df['Rank'] = rank_df.groupby(['Актив', 'Горизонт'])['RMSE'].rank(method='dense', ascending=True)
avg_rank = rank_df.groupby(['Актив', 'Модель', 'Тип'])['Rank'].mean().reset_index()
avg_rank = avg_rank.sort_values(['Актив', 'Rank'])

# График среднего ранга (горизонтальные столбцы)
for group, assets in [('Акции', stocks), ('Криптовалюты', crypto)]:
    df_plot = avg_rank[avg_rank['Актив'].isin(assets)]
    fig = px.bar(
        df_plot,
        x='Rank',
        y='Модель',
        color='Модель',
        orientation='h',
        facet_col='Актив',
        color_discrete_map=color_map,
        title=f'{group} – Средний ранг RMSE (меньше = точнее)',
        labels={'Rank': 'Средний ранг (1 – лучший)'}
    )
    fig.update_layout(template='plotly_white', showlegend=False)
    fig.write_html(os.path.join(output_dir, f'top_models_rank_{group}.html'))

# Таблица "Топ-3 модели по RMSE для каждого актива и горизонта"
top3 = df_all.sort_values(['Актив', 'Горизонт', 'RMSE']).groupby(['Актив', 'Горизонт']).head(3)
top3 = top3[['Актив', 'Горизонт', 'Модель', 'Тип', 'RMSE']].reset_index(drop=True)
top3.to_excel('results_returns/metrics/top3_models_by_RMSE.xlsx', index=False)

print("Готово! Графики с цветами, heatmap и топ-модели сохранены.")

Готово! Графики с цветами, heatmap и топ-модели сохранены.


# Прочее

In [3]:
from funcs.features_create import create_features

series = data['SBER']
X, y = create_features(series, horizon=1)
print("X shape:", X.shape)
print("X index min:", X.index.min(), "max:", X.index.max())
print("Test dates in X:", X.index[X.index >= pd.Timestamp('2025-01-01')][:5])

ml_config = {'X': X, 'y': y, 'model_class': RandomForestRegressor,
             'model_params': {'n_estimators': 200, 'max_depth': 10, 'random_state': 42, 'n_jobs': -1}}

df_pred = walk_forward_validation(series, ml_config=ml_config, test_start='2025-01-01', horizon=1)
print(df_pred.head())
print("Any NaN?", df_pred.isnull().any())

X shape: (2322, 44)
X index min: 2017-01-31 00:00:00 max: 2026-04-29 00:00:00
Test dates in X: DatetimeIndex(['2025-01-03', '2025-01-06', '2025-01-08', '2025-01-09',
               '2025-01-10'],
              dtype='datetime64[ns]', name='date', freq=None)


  H=1: 100%|█████████▉| 334/335 [00:00<00:00, 1274.06window/s]

            actual  predicted
date                         
2025-01-03  274.37        NaN
2025-01-06  277.00        NaN
2025-01-08  271.80        NaN
2025-01-09  278.77        NaN
2025-01-10  279.80        NaN
Any NaN? actual       False
predicted     True
dtype: bool


In [5]:
X.head(5)

,close,close_lag1,close_lag2,close_lag3,close_lag5,close_lag10,close_lag20,return_lag1,return_lag2,return_lag3,...,dow3,dow4,dow5,dow6,close_max5,close_min5,close_max10,close_min10,close_max20,close_min20
date,,,,,,,,,,,,,,,,,,,,,
2017-01-31,172.20,174.60,178.92,173.79,170.25,165.98,180.12,-0.024145,0.029518,0.020733,...,0,0,0,0,178.92,170.26,178.92,165.24,178.92,164.66
2017-02-01,173.90,172.20,174.60,178.92,170.26,165.24,175.95,-0.013746,-0.024145,0.029518,...,0,0,0,0,178.92,172.20,178.92,166.30,178.92,164.66
2017-02-02,172.83,173.90,172.20,174.60,173.79,166.30,171.90,0.009872,-0.013746,-0.024145,...,1,0,0,0,178.92,172.20,178.92,166.75,178.92,164.66
2017-02-03,173.80,172.83,173.90,172.20,178.92,167.49,170.69,-0.006153,0.009872,-0.013746,...,0,1,0,0,174.60,172.20,178.92,166.75,178.92,164.66
2017-02-06,172.87,173.80,172.83,173.90,174.60,166.75,168.35,0.005612,-0.006153,0.009872,...,0,0,0,0,173.90,172.20,178.92,170.25,178.92,164.66


In [ ]:
import pandas as pd
import numpy as np
from funcs.features_create import create_features
from funcs.backtest import walk_forward_validation
from sklearn.ensemble import RandomForestRegressor

series = data['SBER']
X, y = create_features(series, horizon=1)

ml_config = {
    'X': X,
    'y': y,
    'model_class': RandomForestRegressor,
    'model_params': {'n_estimators': 200, 'max_depth': 10, 'random_state': 42, 'n_jobs': -1}
}

# Скопируем функцию с отладочными print (временно)
def debug_walk_forward(series, ml_config, test_start="2025-01-01", horizon=1):
    X_full = ml_config['X']
    y_full = ml_config['y']
    model_class = ml_config['model_class']
    model_params = ml_config['model_params']

    test_start_ts = pd.Timestamp(test_start)
    # Определяем частоту
    from funcs.load_data import get_frequency
    freq = get_frequency(series)
    if freq == 'B':
        test_dates = pd.date_range(test_start_ts, series.index.max(), freq='B')
        test_dates = test_dates[test_dates.isin(series.index)]
    else:
        test_dates = pd.date_range(test_start_ts, series.index.max(), freq='D')
    # Пересекаем с X_full
    test_dates = test_dates[test_dates.isin(X_full.index)]
    print(f"Test dates after filter: {len(test_dates)}")
    print(f"First 5 test dates: {test_dates[:5]}")

    preds, actuals, dates = [], [], []
    i = 0
    while i < len(test_dates):
        current_date = test_dates[i]
        end_idx = min(i + horizon, len(test_dates))
        window_dates = test_dates[i:end_idx]
        print(f"\nWindow {i}: {window_dates}")

        train_end = window_dates[0]
        X_train = X_full[X_full.index < train_end]
        y_train = y_full[y_full.index < train_end]
        print(f"  X_train size: {len(X_train)}, y_train size: {len(y_train)}")

        if len(X_train) == 0:
            forecast = np.full(len(window_dates), np.nan)
            print("  X_train empty -> NaN forecast")
        else:
            model = model_class(**model_params)
            try:
                model.fit(X_train, y_train)
                # Ищем X_window
                X_window = X_full.loc[window_dates]
                print(f"  X_window shape: {X_window.shape}")
                if X_window.shape[0] != len(window_dates):
                    print(f"  Mismatch: X_window={X_window.shape[0]}, window_dates={len(window_dates)}")
                    forecast = np.full(len(window_dates), np.nan)
                else:
                    forecast = model.predict(X_window)
                    print(f"  Forecast: {forecast[:3]}")
                    forecast = np.asarray(forecast).ravel()
            except Exception as e:
                print(f"  Exception: {e}")
                forecast = np.full(len(window_dates), np.nan)

        for j, d in enumerate(window_dates):
            actual = y_full.loc[d] if d in y_full.index else np.nan
            preds.append(forecast[j] if j < len(forecast) else np.nan)
            actuals.append(actual)
            dates.append(d)
        i = end_idx

    results = pd.DataFrame({'date': dates, 'actual': actuals, 'predicted': preds})
    results.set_index('date', inplace=True)
    return results

# Запускаем отладку
df_pred_debug = debug_walk_forward(series, ml_config, test_start="2025-01-01", horizon=1)
print("\nResult head:")
print(df_pred_debug.head(10))
print("Any NaN in predicted?", df_pred_debug['predicted'].isna().any())

Test dates after filter: 334
First 5 test dates: DatetimeIndex(['2025-01-03', '2025-01-06', '2025-01-08', '2025-01-09',
               '2025-01-10'],
              dtype='datetime64[ns]', freq=None)

Window 0: DatetimeIndex(['2025-01-03'], dtype='datetime64[ns]', freq=None)
  X_train size: 1987, y_train size: 1987
  X_window shape: (1, 44)
  Forecast: [270.55364024]

Window 1: DatetimeIndex(['2025-01-06'], dtype='datetime64[ns]', freq=None)
  X_train size: 1988, y_train size: 1988
  X_window shape: (1, 44)
  Forecast: [274.82347038]

Window 2: DatetimeIndex(['2025-01-08'], dtype='datetime64[ns]', freq=None)
  X_train size: 1989, y_train size: 1989
  X_window shape: (1, 44)
  Forecast: [277.57925866]

Window 3: DatetimeIndex(['2025-01-09'], dtype='datetime64[ns]', freq=None)
  X_train size: 1990, y_train size: 1990
  X_window shape: (1, 44)
  Forecast: [273.29040735]

Window 4: DatetimeIndex(['2025-01-10'], dtype='datetime64[ns]', freq=None)
  X_train size: 1991, y_train size: 1991
  X_

In [ ]:
# new version

import warnings
from statsmodels.tools.sm_exceptions import ValueWarning
warnings.filterwarnings('ignore', category=ValueWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Горизонты
# HORIZONS = (1, 7, 21)
HORIZONS = (1,)

# Словари для хранения результатов
all_results = {}       # (asset, model) -> результаты прогнозов
all_metrics = {}       # (asset, model) -> метрики
all_backtests = {}     # (asset, model, horizon) -> DataFrame бэктеста

# Для каждого актива
# for asset_name in data:
for i in range(1):
    series = data[asset_name]
    freq = get_frequency(series)
    
    seasonal_periods = 5 if freq == 'B' else 7  # для акций 5, для крипто 7
    print(f"\n=== {asset_name} (частота {freq}, сезонный период {seasonal_periods}) ===")

    # Модели
    models = {
        'SMA_20': lambda train, steps: sma_predict(train, steps, window=20),
        'ETS': lambda train, steps: ets_predict(train, steps, seasonal_periods=seasonal_periods),
        # 'SARIMA': lambda train, steps: sarima_predict(train, steps, seasonal_periods=seasonal_periods),
    }
    for model_name, model_func in models.items():
        print(f"\n  Модель: {model_name}")
        for h in HORIZONS:
            df_pred = walk_forward_validation(
                series, model_func, test_start=TEST_START, horizon=h, logs=False
            )
            all_results[(asset_name, model_name, h)] = df_pred
            df_pred.to_excel(f'results/1_{model_name}/predictions/{asset_name}_preds_{model_name}_{h}.xlsx')

            metrics = evaluate_metrics(df_pred)
            all_metrics[(asset_name, model_name, h)] = metrics
            print(f"      RMSE={metrics['RMSE']:.4f}, MAE={metrics['MAE']:.4f}, MAPE={metrics['MAPE']:.2f}%")

            bt = Backtester(
                price_series=series,
                predictions_df=all_results[(asset_name, model_name, h)],
                initial_capital=100_000,
                threshold=0.005,
                buy_fraction=0.1,      # 10% от свободных денег на каждую покупку
                commission_rate=0.001  # 0.1%
            )
            results_df, trades_df, fig = bt.run()

            results_df.to_excel(f'results/1_{model_name}/backtest/{asset_name}_backtest_{model_name}_{h}.xlsx')
            trades_df.to_excel(f'results/1_{model_name}/backtest/{asset_name}_trades_{model_name}_{h}.xlsx')
            fig.write_html(f'results/1_{model_name}/plots/{asset_name}_{model_name}_{h}.html')

            # fig.show()



# df['prev_close'] = df['close'].shift(1)                # цена предыдущего дня


=== ETH (частота D, сезонный период 7) ===

  Модель: SMA_20


  H=1: 100%|██████████| 485/485 [00:00<00:00, 2964.73window/s]

      RMSE=287.4687, MAE=216.5390, MAPE=7.86%


In [16]:
df_res_all = all_results[(asset_name, model_name, h)]
df_res_all.head(10)

,actual,predicted
date,,
2025-01-01,3353.504150,3555.524207
2025-01-02,3451.392578,3529.044336
2025-01-03,3605.009766,3506.053699
2025-01-04,3657.706787,3492.883923
2025-01-05,3634.103760,3478.172205
2025-01-06,3688.611328,3460.503345
2025-01-07,3381.577393,3450.595605
2025-01-08,3326.329346,3438.734912
2025-01-09,3219.430664,3434.154980


In [ ]:
df_res_all[]

In [17]:
share = 'SBER'
model = 'SMA_20'
h = 7

bt = Backtester(
    price_series=data['SBER'],
    predictions_df=all_results[('SBER', 'SMA_20', 7)],
    initial_capital=100_000,
    threshold=0.005,
    buy_fraction=0.1,      # 10% от свободных денег на каждую покупку
    commission_rate=0.001  # 0.1%
)
results_df, trades_df, fig = bt.run()

results_df.to_excel(f'results/1_{model}/{share}_backtest_{model}_{h}.xlsx')
trades_df.to_excel(f'results/1_{model}/backtest/{share}_trades_{model}_{h}.xlsx')
fig.write_html(f'results/1_{model}/plots/{share}_{model}_{h}.html')

fig.show()

In [14]:
results_df

,close,predicted,prev_close,expected_change,equity,shares
date,,,,,,
2025-01-03,272.25,245.931000,NaN,NaN,100000.000000,0.0
2025-01-06,274.37,247.000050,272.25,-0.092745,100000.000000,0.0
2025-01-08,277.00,247.676053,274.37,-0.097292,100000.000000,0.0
2025-01-09,271.80,248.167855,277.00,-0.104087,100000.000000,0.0
2025-01-10,278.77,248.711748,271.80,-0.084946,100000.000000,0.0
...,...,...,...,...,...,...
2026-04-24,325.19,318.802250,327.39,-0.026231,107860.315213,0.0
2026-04-27,321.99,319.048363,325.19,-0.018886,107860.315213,0.0
2026-04-28,319.82,319.276781,321.99,-0.008426,107860.315213,0.0


In [ ]:
class Backtester:
    """
    Гибкий бэктестер торговой стратегии.
    - Постепенный вход (покупка на фиксированную долю свободных денег при сигнале "рост").
    - Полный выход (продажа всех акций при сигнале "падение").
    - Учитывает комиссию за каждую сделку.
    - Строит 3-панельный график: цена + сигналы, капитал, количество акций.
    """
    def __init__(self, price_series, predictions_df,
                 initial_capital=100_000, threshold=0.005,
                 buy_fraction=0.1, commission_rate=0.001):
        """
        Параметры
        ----------
        price_series : pd.Series с DatetimeIndex
            Фактические цены закрытия.
        predictions_df : pd.DataFrame
            Должен содержать колонку 'predicted', индекс — даты как у price_series.
            Прогноз на дату t находится в той же строке, что и фактическая цена t.
        initial_capital : float
            Стартовый капитал (например, 100 000 долларов).
        threshold : float
            Порог ожидаемого изменения для входа/выхода.
            Например, 0.005 означает "если прогноз обещает рост >0.5%, покупаем".
        buy_fraction : float
            Доля свободных денежных средств, используемая при каждой покупке.
            0.1 = 10%. При повторных сигналах позиция наращивается.
        commission_rate : float
            Комиссия в долях от суммы сделки (0.001 = 0.1%).
        """
        self.price = price_series
        self.pred = predictions_df
        self.cash = initial_capital          # начальные свободные деньги
        self.threshold = threshold
        self.buy_size = buy_fraction         # доля капитала на одну покупку
        self.comm = commission_rate
        self.results = None                  # здесь будет DataFrame после run()
        self.trades = []                     # список словарей с деталями сделок

    def run(self):
        """
        Запускает симуляцию торговли по дням.
        Возвращает:
            results_df (pd.DataFrame) – колонки: close, predicted, equity, shares
            trades_df (pd.DataFrame) – таблица всех сделок
            figure (plotly Figure) – график
        """
        # Берём только те даты, где есть и цена, и прогноз
        common_idx = self.price.index.intersection(self.pred.index)
        df = pd.DataFrame(index=common_idx)
        df['close'] = self.price.loc[common_idx]               # фактическая цена
        df['predicted'] = self.pred.loc[common_idx, 'predicted'] # прогноз на этот же день
        df['prev_close'] = df['close'].shift(1)                # цена предыдущего дня
        # Ожидаемое изменение: (прогноз[t] - цена[t-1]) / цена[t-1]
        df['expected_change'] = (df['predicted'] - df['prev_close']) / df['prev_close']

        cash = self.cash
        shares = 0.0                        # текущее количество акций
        invested_total = 0.0                # сумма, вложенная в текущую открытую позицию (с учётом комиссий)
        equity_hist = []                    # стоимость портфеля на каждый день
        shares_hist = []                    # количество акций на каждый день
        trades_list = []                    # список сделок
        position_open = False               # флаг: есть ли открытая позиция

        # Цикл по дням
        for i, (idx, row) in enumerate(df.iterrows()):
            # Для первого дня или если нет предыдущей цены — просто запоминаем текущее состояние
            if pd.isna(row['prev_close']) or pd.isna(row['expected_change']):
                portfolio_val = cash + shares * row['close']
                equity_hist.append(portfolio_val)
                shares_hist.append(shares)
                continue

            # Определяем торговый сигнал
            signal = 0
            if row['expected_change'] > self.threshold:         # ожидаем рост выше порога
                signal = 1                                       # сигнал на покупку
            elif row['expected_change'] < -self.threshold and position_open:
                signal = -1                                      # сигнал на продажу (только если есть позиция)

            # ========== ПОКУПКА (частичная) ==========
            if signal == 1:
                # Сколько свободных денег готовы потратить
                buy_amount = cash * self.buy_size
                if buy_amount > 0:
                    commission = self.comm * buy_amount          # комиссия с суммы сделки
                    cost = buy_amount - commission               # чистая сумма, на которую покупаем
                    price = row['close']                         # цена покупки (тек. закрытие)
                    new_shares = cost / price                    # количество купленных акций
                    shares += new_shares                         # увеличиваем общее количество
                    cash -= (cost + commission)                  # списываем деньги (включая комиссию)
                    invested_total += (cost + commission)        # добавляем в общую вложенную сумму
                    position_open = True
                    trades_list.append({
                        'type': 'buy',
                        'date': idx,
                        'price': price,
                        'shares': new_shares,
                        'commission': commission,
                        'value': cost                            # чистая стоимость покупки
                    })

            # ========== ПРОДАЖА (всей позиции) ==========
            elif signal == -1 and position_open:
                if shares > 0:
                    price = row['close']
                    gross = shares * price                       # выручка до комиссии
                    commission = self.comm * gross
                    net = gross - commission                     # чистая выручка
                    cash += net                                  # деньги возвращаются на счёт
                    trades_list.append({
                        'type': 'sell',
                        'date': idx,
                        'price': price,
                        'shares': shares,
                        'commission': commission,
                        'proceeds': net,
                        'total_invested': invested_total,
                        # доходность сделки в процентах
                        'return_pct': (net - invested_total) / invested_total * 100
                    })
                    shares = 0.0
                    invested_total = 0.0
                    position_open = False

            # Сохраняем состояние портфеля на конец дня
            portfolio_val = cash + shares * row['close']        # ← вот это и есть "капитал"
            equity_hist.append(portfolio_val)
            shares_hist.append(shares)

        # ========== Закрытие позиции в конце периода, если осталась открытой ==========
        if position_open and shares > 0:
            last_price = df['close'].iloc[-1]                   # последняя доступная цена
            gross = shares * last_price
            commission = self.comm * gross
            net = gross - commission
            cash += net
            portfolio_val = cash                                # после продажи все деньги в кэше
            equity_hist[-1] = portfolio_val                     # корректируем последнее значение
            trades_list.append({
                'type': 'sell',
                'date': df.index[-1],
                'price': last_price,
                'shares': shares,
                'commission': commission,
                'proceeds': net,
                'total_invested': invested_total,
                'return_pct': (net - invested_total) / invested_total * 100
            })
            shares_hist[-1] = 0.0                               # акций больше нет

        # Сохраняем результаты в DataFrame
        df['equity'] = equity_hist
        df['shares'] = shares_hist
        self.results = df
        self.trades = pd.DataFrame(trades_list)
        return df, self.trades, self._plot()

    def _plot(self):
        """Строит 3-панельный график: цена, капитал, количество акций."""
        df = self.results
        # Таблицы покупок и продаж для маркеров
        buy_df = self.trades[self.trades['type'] == 'buy'] if not self.trades.empty else pd.DataFrame()
        sell_df = self.trades[self.trades['type'] == 'sell'] if not self.trades.empty else pd.DataFrame()

        fig = make_subplots(
            rows=3, cols=1, shared_xaxes=True,
            subplot_titles=("Цена и сигналы", "Капитал", "Количество акций"),
            row_heights=[0.5, 0.25, 0.25],
            vertical_spacing=0.08
        )

        # --- Панель 1: Фактическая цена, прогноз и точки сделок ---
        fig.add_trace(go.Scatter(
            x=df.index, y=df['close'], name='Факт', line=dict(color='black')
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=df.index, y=df['predicted'], name='Прогноз',
            line=dict(color='blue', dash='dot')
        ), row=1, col=1)
        if not buy_df.empty:
            fig.add_trace(go.Scatter(
                x=buy_df['date'], y=buy_df['price'], mode='markers',
                name='Покупка', marker=dict(color='green', size=8, symbol='triangle-up')
            ), row=1, col=1)
        if not sell_df.empty:
            fig.add_trace(go.Scatter(
                x=sell_df['date'], y=sell_df['price'], mode='markers',
                name='Продажа', marker=dict(color='red', size=8, symbol='triangle-down')
            ), row=1, col=1)

        # --- Панель 2: Кривая капитала ---
        fig.add_trace(go.Scatter(
            x=df.index, y=df['equity'], name='Капитал', line=dict(color='purple')
        ), row=2, col=1)
        # Горизонтальная линия стартового капитала
        fig.add_hline(y=self.cash, line_dash="dash", line_color="gray", row=2, col=1)

        # --- Панель 3: Количество акций в портфеле ---
        fig.add_trace(go.Scatter(
            x=df.index, y=df['shares'], name='Акции', fill='tozeroy',
            line=dict(color='orange')
        ), row=3, col=1)
        fig.update_yaxes(title_text="Кол-во", row=3, col=1)

        fig.update_layout(height=900, title="Бэктест стратегии")
        fig.update_xaxes(title_text="Дата", row=3, col=1)
        return fig

In [4]:
for asset_name in data:
    series = data[asset_name]
    for model_name in models:
        for h in HORIZONS:
            pred_df = all_results[(asset_name, model_name)][h]
            if not pred_df.empty:
                bt = backtest_trading(series, pred_df, threshold=THRESHOLD)
                all_backtests[(asset_name, model_name, h)] = bt
                print(f"{asset_name} {model_name} H={h}: "
                      f"Итоговая доходность {bt['equity'].iloc[-1]-1:.2%}")

SBER SMA_20 H=1: Итоговая доходность 17.72%
SBER SMA_20 H=7: Итоговая доходность 30.22%
SBER SMA_20 H=21: Итоговая доходность 28.49%
YNDX SMA_20 H=1: Итоговая доходность 17.39%
YNDX SMA_20 H=7: Итоговая доходность 31.15%
YNDX SMA_20 H=21: Итоговая доходность 38.93%
BTC SMA_20 H=1: Итоговая доходность -8.54%
BTC SMA_20 H=7: Итоговая доходность 9.01%
BTC SMA_20 H=21: Итоговая доходность 11.87%
ETH SMA_20 H=1: Итоговая доходность -19.94%
ETH SMA_20 H=7: Итоговая доходность 52.63%
ETH SMA_20 H=21: Итоговая доходность 58.18%


In [13]:
bt

,close,predicted,expected_change,daily_return,equity
date,,,,,
2025-01-01,3353.504150,2314.197144,-0.309917,0.000000,1.000000
2025-01-02,3451.392578,2313.508478,-0.329688,0.000000,1.000000
2025-01-03,3605.009766,2312.785379,-0.358452,0.000000,1.000000
2025-01-04,3657.706787,2309.620241,-0.368561,0.000000,1.000000
2025-01-05,3634.103760,2306.296846,-0.365374,0.000000,1.000000
...,...,...,...,...,...
2026-04-26,2369.729004,2297.206095,-0.030604,0.000000,0.640084
2026-04-27,2303.063965,2299.253838,-0.001654,0.000000,0.640084
2026-04-28,2289.419678,2301.403969,0.005235,-0.005924,0.636292


In [4]:
# Пример для одного актива и модели (можно изменить на цикл или выбор)
asset_name = "SBER"
model_name = "SARIMA"
results = all_results[(asset_name, model_name)]
plot_forecast_vs_actual(results, asset_name, model_name)

In [5]:
summary = []
for (asset, model), met in all_metrics.items():
    summary.append({
        'Актив': asset,
        'Модель': model,
        'RMSE': met[1]['RMSE'],
        'MAE': met[1]['MAE'],
        'MAPE': met[1]['MAPE']
    })
summary_df = pd.DataFrame(summary)
summary_df.sort_values(['Актив', 'RMSE']).style.background_gradient(subset=['RMSE', 'MAE', 'MAPE'], cmap='RdYlGn_r')

,Актив,Модель,RMSE,MAE,MAPE
7,BTC,ETS,2120.146905,1529.651069,1.661190
8,BTC,SARIMA,2128.649356,1535.081393,1.667023
6,BTC,SMA_20,5140.522738,3971.272403,4.338219
10,ETH,ETS,112.487822,77.292514,2.702065
11,ETH,SARIMA,112.647889,77.727250,2.714961
9,ETH,SMA_20,287.468677,216.538953,7.859869
2,SBER,SARIMA,3.814051,2.639609,0.871256
1,SBER,ETS,3.833251,2.651022,0.874999
0,SBER,SMA_20,9.043790,6.521924,2.158489
4,YNDX,ETS,72.282457,55.118454,1.303406


In [6]:
asset_name = "SBER"
fig = go.Figure()
# Добавляем модели (H=1)
for model_name in models.keys():
    bt = all_backtests.get((asset_name, model_name, 1))
    if bt is not None:
        fig.add_trace(go.Scatter(x=bt.index, y=bt['equity'],
                                 name=f"{model_name} (H=1)"))
# Buy & Hold
close_series = data[asset_name]
close_series_test = close_series[TEST_START:]
fig.add_trace(go.Scatter(x=close_series_test.index,
                         y=close_series_test / close_series_test.iloc[0],
                         name='Buy & Hold', line=dict(dash='dot')))
fig.update_layout(title=f"Сравнение стратегий для {asset_name}")
fig.show()